# 投资者行为分析

本Notebook用于分析交易所债券投资者结构数据。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime
import json
import random

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入网络请求库
import requests

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine

# 导入配置
from config import DATABASE_URL, SSE_URL_TEMPLATE, SZSE_URL_TEMPLATE, CCDC_URL, SHCH_URL, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 数据库连接

In [ ]:
def get_db_connection():
    """获取数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

engine, conn = get_db_connection()

## 3. 上交所数据获取

In [ ]:
def fetch_sse_data(year, month):
    """
    获取上交所投资者结构数据
    
    参数:
        year: 年份
        month: 月份
    """
    trade_date = f'{year}-{month}'
    timestamp = int(datetime.datetime.now().timestamp() * 1000)
    
    url = SSE_URL_TEMPLATE.format(trade_date=trade_date, timestamp=timestamp)
    
    headers = {
        'Accept': '*/*',
        'Accept-Encoding': 'gzip, deflate',
        'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
        'Connection': 'keep-alive',
        'Host': 'query.sse.com.cn',
        'Referer': 'http://bond.sse.com.cn/',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    post_dict = {
        'jsonCallBack': 'jsonpCallback73603',
        'isPagination': 'false',
        'sqlId': 'COMMON_BOND_SCSJ_SCTJ_TJYB_CYJG_L',
        'TRADEDATE': trade_date,
        '_': str(timestamp)
    }
    
    try:
        response = requests.post(url, headers=headers, json=post_dict, timeout=20)
        content = response.content.decode('utf-8')
        # 解析JSONP响应
        json_str = content[len('jsonpCallback73603('):-1]
        data = json.loads(json_str)
        df = pd.DataFrame(data['result'])
        return df
    except Exception as e:
        print(f"获取上交所数据失败: {e}")
        return None

# 获取当前月份数据
now = datetime.datetime.now()
sse_df = fetch_sse_data(now.year, now.month)

if sse_df is not None:
    print(f"上交所数据获取成功，共 {len(sse_df)} 条记录")
    display(sse_df.head())

## 4. 深交所数据获取

In [ ]:
def fetch_szse_data(year, month):
    """
    获取深交所投资者结构数据
    
    参数:
        year: 年份
        month: 月份
    """
    random_num = round(random.random(), 15)
    trade_date = f'{year}-{month:02d}'
    
    url = SZSE_URL_TEMPLATE.format(trade_date=trade_date, random=random_num)
    
    headers = {
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept-Language': 'zh-CN,zh;q=0.9',
        'Connection': 'keep-alive',
        'Content-Type': 'application/json',
        'Host': 'bond.szse.cn',
        'Referer': 'https://bond.szse.cn/marketdata/statistics/report/struc/index.html',
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'X-Request-Type': 'ajax',
        'X-Requested-With': 'XMLHttpRequest'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=20)
        data = json.loads(response.text)
        
        # 解析日期
        dt = data[0]['metadata']['conditions'][0]['defaultValue']
        dt = pd.to_datetime(dt, format='%Y-%m')
        dt = dt + pd.offsets.MonthEnd(1)
        
        # 提取表格数据
        table_data = data[0]['data']
        df = pd.DataFrame(table_data)
        df['trade_date'] = dt.strftime('%Y-%m-%d')
        
        return df
    except Exception as e:
        print(f"获取深交所数据失败: {e}")
        return None

# 获取深交所数据
szse_df = fetch_szse_data(now.year, now.month)

if szse_df is not None:
    print(f"深交所数据获取成功，共 {len(szse_df)} 条记录")
    display(szse_df.head())

## 5. 数据分析与可视化

In [ ]:
# 数据清洗函数
def clean_numeric_column(col):
    """清洗数值列"""
    try:
        return col.astype(str).str.replace(',', '').astype(float)
    except:
        return col

# 清洗上交所数据
if sse_df is not None:
    for col in sse_df.columns:
        if sse_df[col].dtype == 'object':
            sse_df[col] = clean_numeric_column(sse_df[col])
    
    print("上交所数据清洗完成")
    display(sse_df.describe())

In [ ]:
# 可视化投资者结构
if sse_df is not None and 'TYPE' in sse_df.columns and 'HOLD_PERCENT' in sse_df.columns:
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # 按持仓比例排序
    plot_df = sse_df.sort_values('HOLD_PERCENT', ascending=True)
    
    ax.barh(plot_df['TYPE'], plot_df['HOLD_PERCENT'], color='steelblue')
    ax.set_xlabel('持仓比例 (%)')
    ax.set_ylabel('投资者类型')
    ax.set_title(f'上交所债券投资者结构 ({now.year}-{now.month})')
    
    plt.tight_layout()
    os.makedirs('output', exist_ok=True)
    plt.savefig('output/sse_investor_structure.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 数据库存储

In [ ]:
# 存储上交所数据
if sse_df is not None and conn is not None:
    try:
        sse_df.to_sql('investor_sh', con=conn, if_exists='append', index=False)
        print(f"上交所数据已存入数据库，共 {len(sse_df)} 条记录")
    except Exception as e:
        print(f"存储上交所数据失败: {e}")

# 存储深交所数据
if szse_df is not None and conn is not None:
    try:
        szse_df.to_sql('investor_sz', con=conn, if_exists='append', index=False)
        print(f"深交所数据已存入数据库，共 {len(szse_df)} 条记录")
    except Exception as e:
        print(f"存储深交所数据失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成输出报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

print(f"\n数据获取时间: {now.strftime('%Y-%m-%d %H:%M:%S')}")

if sse_df is not None:
    print(f"\n上交所数据:")
    print(f"  - 记录数: {len(sse_df)}")
    print(f"  - 列数: {len(sse_df.columns)}")

if szse_df is not None:
    print(f"\n深交所数据:")
    print(f"  - 记录数: {len(szse_df)}")
    print(f"  - 列数: {len(szse_df.columns)}")

# 保存处理后的数据
os.makedirs('output', exist_ok=True)

if sse_df is not None:
    sse_df.to_excel('output/sse_investor_data.xlsx', index=False)
    print(f"\n上交所数据已保存至: output/sse_investor_data.xlsx")

if szse_df is not None:
    szse_df.to_excel('output/szse_investor_data.xlsx', index=False)
    print(f"深交所数据已保存至: output/szse_investor_data.xlsx")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if conn is not None:
    conn.close()
    print("数据库连接已关闭")